In [ ]:
# ✅ 1. Check GPU
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

True
Tesla T4


In [ ]:
# ✅ 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
# from transformers import AutoTokenizer, AutoModelForCausalLM

# model_name = "mistralai/Mistral-7B-Instruct-v0.2"
# model_path = "/content/drive/My Drive/mistral-7b"

# # Download tokenizer + model permanently into your Drive
# tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_path)
# model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_path, torch_dtype="auto").cuda()


In [ ]:
# ✅ 3. Load Model
model_path = "/content/drive/My Drive/mistral-7b/models--mistralai--Mistral-7B-Instruct-v0.2/snapshots/3ad372fc79158a2148299e3318516c786aeded6c"
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(model_path, legacy=True)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto").cuda()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
!pip install pyngrok --quiet
from pyngrok import ngrok
ngrok.kill()  # Kill all tunnels

import os
os.system("fuser -k 5000/tcp")  # Kill anything using port 5000
os.system("pkill -f ngrok")     # Force kill ngrok processes


15

In [ ]:
!pip install pyngrok --quiet
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading
import os

# Kill anything on port 5000 (important if runtime was interrupted)
os.system("fuser -k 5000/tcp || true")

# Flask App
app = Flask(__name__)

@app.route("/generate", methods=["POST"])
def generate():
    prompt = request.json.get("prompt", "")
    max_tokens = request.json.get("max_tokens", 100)  # <-- default to 100 if not sent
    if not prompt:
        return jsonify({"error": "No prompt provided"}), 400
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_tokens)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    decoded = decoded[len(prompt):].strip()
    return jsonify({"response": decoded})


# Kill ngrok tunnels and start fresh
ngrok.kill()
ngrok.set_auth_token("2zUi49U7JYa5S123DbFN2shwYrY_5rCmSWQmWrNmEQ4qQeWet")
public_url = ngrok.connect(addr="5000")
print("🚀 Public URL:", public_url.public_url)

# Start Flask in background
def run_flask():
    app.run(port=5000)

thread = threading.Thread(target=run_flask)
thread.start()


🚀 Public URL: https://6550f83a1e3f.ngrok-free.app


In [ ]:
import requests

test_prompt = (
    "Respond only in JSON format. List 5 popular books along with their authors. "
    "Structure it like this: "
    "{\"books\": [{\"title\": \"Book Title\", \"author\": \"Author Name\"}, ...]}"
)

res = requests.post(public_url.public_url + "/generate", json={"prompt": test_prompt})
print(res.status_code)
print(res.text)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
INFO:werkzeug:127.0.0.1 - - [10/Jul/2025 09:59:44] "POST /generate HTTP/1.1" 200 -


200
{"response":"{\n  \"books\": [\n    {\n      \"title\": \"To Kill a Mockingbird\",\n      \"author\": \"Harper Lee\"\n    },\n    {\n      \"title\": \"1984\",\n      \"author\": \"George Orwell\"\n    },\n    {\n      \"title\": \"Pride and Prejudice\",\n      \"author\": \"Jane Austen\"\n    },\n    {"}

